## MarketWatch Scraper

In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import duckdb as dd
from datetime import datetime
import time


DB_PATH = "marketwatch_news.duckdb"
headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Origin': 'https://www.marketwatch.com',
        'Referer': 'https://www.marketwatch.com/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-User': '?1',
        'Sec-Fetch-Dest': 'document',
        'Sec-CH-UA': '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        'Sec-CH-UA-Mobile': '?0',
        'Sec-CH-UA-Platform': '"Windows"',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    }
delay_seconds = float(1.5)
session = requests.Session()
session.headers.update(headers)
_session_primed = False

def init_db(db_path: str = DB_PATH):
    conn = dd.connect(db_path)
    conn.execute(
        '''
        DROP TABLE IF EXISTS marketwatch_articles;

        CREATE TABLE IF NOT EXISTS marketwatch_articles (
            query TEXT,
            title TEXT,
            url TEXT,
            subhead TEXT,
            snippet TEXT,
            author TEXT,
            published_at TIMESTAMP,
            last_updated_at TIMESTAMP,
            scraped_at TIMESTAMP
        );
        '''
    )
    conn.close()


def build_search_url(query: str) -> str:
    """
    Use MarketWatch search, sorted by recency.
    This targets both tickers and company names.
    """
    q = quote_plus(query)
    # Basic search URL; you can refine with more params if needed
    ## return f"https://www.marketwatch.com/search?q={q}&page={page}&sort=recency"
    return f"https://www.marketwatch.com/search?q={q}&ts=2&m=0&sm=0&tab=All%20News"


def prime_session():
    global _session_primed
    if _session_primed:
        return
    try:
        resp = session.get("https://www.marketwatch.com/", timeout=15)
        resp.raise_for_status()
    except requests.RequestException:
        pass
    _session_primed = True


def fetch_search_page(query: str) -> BeautifulSoup:
    global session, _session_primed
    url = build_search_url(query)
    prime_session()

    # Use the global session so cookies and headers persist between homepage
    # requests and search requests. This reduces the chance of a 401 Forbidden.
    session.headers.update({
        "Referer": "https://www.marketwatch.com/",
        "Origin": "https://www.marketwatch.com",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-User": "?1",
        "Sec-Fetch-Dest": "document",
        "Sec-CH-UA": '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        "Sec-CH-UA-Mobile": "?0",
        "Sec-CH-UA-Platform": '"Windows"',
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    })

    try:
        session.get("https://www.marketwatch.com/", timeout=15)
    except requests.RequestException:
        pass

    resp = session.get(url, timeout=15, allow_redirects=True)
    if resp.status_code == 401:
        # If the site still returns 401, reset the global session and retry
        # with a fresh browser-like session and homepage cookies.
        session = requests.Session()
        session.headers.update(headers)
        _session_primed = False
        prime_session()
        try:
            session.get("https://www.marketwatch.com/", timeout=15)
        except requests.RequestException:
            pass
        resp = session.get(url, timeout=15, allow_redirects=True)

    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def fetch_article_page(url: str) -> BeautifulSoup:
    prime_session()
    if not url.startswith("http"):
        url = f"https://www.marketwatch.com{url}"
    session.headers.update({"Referer": "https://www.marketwatch.com/"})
    resp = session.get(url, timeout=15, allow_redirects=True)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def parse_search_results(soup: BeautifulSoup, delay_seconds: float = delay_seconds):
    """
    Parse MarketWatch search results and article pages.

    NOTE: MarketWatch may change its HTML structure.
    Inspect the page in your browser and adjust selectors if needed.
    """
    articles = []

    # Common pattern: result cards under something like:
    # <div class="searchresult"> or <div class="article__content"> etc.
    # We'll try a couple of patterns.
    result_containers = soup.select("div.searchresult, div.article__content")

    for container in result_containers:
        # Title and URL
        a = container.find("a", href=True)
        if not a:
            continue

        title = a.get_text(strip=True)
        url = str(a["href"])
        subhead = None
        snippet = None
        author = None
        published_at = None
        last_updated_at = None
        try:
            article_page = fetch_article_page(url)

            # Subhead
            subhead_el = article_page.select_one('h2[class*="article__subhead"]')
            subhead = subhead_el.get_text(strip=True) if subhead_el else None

            # Snippet
            snippet_el = article_page.select_one('section[class*="Container"]')
            snippet = snippet_el.get_text(strip=True) if snippet_el else None

            # Author
            author_el = article_page.select_one('a[data-testid*="author-link"]')
            author = author_el.get_text(strip=True) if author_el else None

            #Published Timestamp
            published_time_el = article_page.select_one('div[class*="article__timestamp"]>span[class*="first"]>time')
            published_at = published_time_el['datetime'] if published_time_el else None

            #Last Updated Timestamp
            last_updated_time_el = article_page.select_one('div[class*="article__timestamp"]>span[class*="last"]>time')
            last_updated_at = last_updated_time_el['datetime'] if last_updated_time_el else None
        
        except:
            # If fetching/parsing article page fails, we can still save basic info
            pass

        articles.append(
            {
                "title": title,
                "url": url,
                "subhead": subhead,
                "snippet": snippet,
                "author": author,
                "published_at": published_at,
                "last_updated_at": last_updated_at
            }
        )

        time.sleep(delay_seconds)  # be polite

    return articles


def save_articles_to_duckdb(
    query: str,
    articles,
    db_path: str = DB_PATH,
):
    conn = dd.connect(db_path)
    scraped_at = datetime.utcnow()
    
    for art in articles:
        conn.execute(
            """
            INSERT INTO marketwatch_articles
                (query, title, url, subhead, snippet, author, published_at, last_updated_at, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            [
                query,
                art.get("title"),
                art.get("url"),
                art.get("subhead"),
                art.get("snippet"),
                art.get("author"),
                art.get("published_at"),
                art.get("last_updated_at"),
                scraped_at,
            ],
        )

    conn.close()


def scrape_marketwatch_for_query(
    query: str,
    pages: int = 1,
    delay_seconds: float = delay_seconds,
):
    """
    Scrape MarketWatch search results for a ticker or company name
    and store them into DuckDB.
    """
    all_articles = []

    for page in range(1, pages + 1):
        print(f"Fetching page {page} for query '{query}'...")
        soup = fetch_search_page(query)
        articles = parse_search_results(soup)
        print(f"  Found {len(articles)} articles on page {page}.")
        if articles not in all_articles:
            all_articles.extend(articles)
        time.sleep(delay_seconds)  # be polite
    
    save_articles_to_duckdb(query, all_articles)  
    print(f"Saved {len(all_articles)} articles for query '{query}'.")


## Run Main Program

In [11]:
if __name__ == "__main__":
    # 1. Initialize DB (run once)
    init_db()

    # 2. Example: scrape daily news for a ticker
    #    You can schedule this script via cron/Task Scheduler to run daily.
    tickers = ['TSLA'] ##, 'MSFT', 'AAPL', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK.A', 'V', 'UNH', 'HD', 'PG', 'MA', 'DIS', 'CMCSA', 'PFE' , 'VZ', 'KO', 'MRK', 'IBM']          # or "MSFT", "TSLA", etc.
    ## company_name = "Microsoft"   # you can also use company names

    # Scrape by ticker
    for ticker in tickers:
        scrape_marketwatch_for_query(ticker)

    # Scrape by company name (optional)
    ## scrape_marketwatch_for_query(company_name, pages=1)

Fetching page 1 for query 'TSLA'...


HTTPError: 401 Client Error: HTTP Forbidden for url: https://www.marketwatch.com/search?q=TSLA&ts=2&m=0&sm=0&tab=All%20News

## Query Results

In [7]:
## import duckdb as dd
con = dd.connect(DB_PATH)  # Open the persistent database
result = con.execute('''
            -- Parse many formats and never throw
            CREATE OR REPLACE MACRO SAFE_PARSE(ts_text) AS
            coalesce(
            strptime(ts_text, '%B %d, %Y %-I:%M %p'),
            '1900-01-01 00:00:00'
            );

            SELECT *
            --SAFE_PARSE(REPLACE(REPLACE(REPLACE(REPLACE(last_updated_at, 'a.m.', 'AM'), 'p.m.', 'PM'), 'at', ''), 'ET', '')) AS last_updated_at_parsed
            FROM marketwatch_articles
            --WHERE (url LIKE '%barrons.com%' or url LIKE '%marketwatch.com%')
            ;
            '''
).df()
## result = con.sql("SELECT *, SAFE_PARSE(last_updated_at) AS last_updated_at_parsed FROM marketwatch_articles --WHERE (url LIKE '%barrons.com%' or url LIKE '%marketwatch.com%')").df()  # Query results
con.close()
result

,query,title,url,subhead,snippet,author,last_updated_at,scraped_at
0,MSFT,A bullish indicator for software stocks just f...,https://www.marketwatch.com/story/a-bullish-in...,After a five-month slide fueled by AI disrupti...,"For the first time since November, software in...",Christine Ji,NaT,2026-04-16 16:04:09.026413
1,MSFT,Why this market rally still has room to run — ...,https://www.marketwatch.com/story/why-this-mar...,Nomura strategist says energy shortages could ...,"The S&P 500 is back in record-setting mode, a...",Barbara Kollmeyer,2026-04-16 12:51:00,2026-04-16 16:04:09.026413
2,MSFT,Microsoft’s stock has sprung back to life — an...,https://www.marketwatch.com/story/microsofts-s...,"After frustrating investors for months, Micros...",Microsoft’s stock still has a ways to go befor...,Hannah Pedone,NaT,2026-04-16 16:04:09.026413
3,MSFT,April 15 is tax deadline day — and here’s why ...,https://www.marketwatch.com/story/april-15-is-...,Less money flowing through the market isn’t a ...,Happy Tax Day to those who celebrate. Or don’t...,Barbara Kollmeyer,2026-04-15 13:44:00,2026-04-16 16:04:09.026413
4,MSFT,Why one analyst believes Microsoft’s stock may...,https://www.marketwatch.com/story/why-one-anal...,Bernstein’s Mark Moerdler sees a good entry po...,Investors have been waiting for Microsoft’s en...,Hannah Pedone,2026-04-15 03:02:00,2026-04-16 16:04:09.026413
...,...,...,...,...,...,...,...,...
72,MSFT,Will AI start ‘going rogue’? The chorus of war...,https://www.marketwatch.com/story/will-ai-star...,NaN,NaN,NaN,NaT,2026-04-16 16:04:09.026413
73,MSFT,Intel’s stock hasn’t been this hot in 38 years...,https://www.marketwatch.com/story/intels-stock...,NaN,NaN,NaN,NaT,2026-04-16 16:04:09.026413
74,MSFT,This might be the best time for you to load up...,https://www.marketwatch.com/story/this-might-b...,NaN,NaN,NaN,NaT,2026-04-16 16:04:09.026413
75,MSFT,,#,NaN,NaN,NaN,NaT,2026-04-16 16:04:09.026413
